In [1]:
!pip install opendatasets

## 1. Downloading Dataset directly from Kaggle

In [4]:
import opendatasets as od

od.download('https://www.kaggle.com/datasets/amineipad/e-commerce-marketing-and-sales-revenue-prediction')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: faisal_altaf_06
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/amineipad/e-commerce-marketing-and-sales-revenue-prediction


100%|██████████| 1.00M/1.00M [00:00<00:00, 550MB/s]

## 📦 2. Load E-Commerce Marketing & Sales Dataset

In [28]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt




# Load Titanic dataset
df = pd.read_csv("/content/e-commerce-marketing-and-sales-revenue-prediction/train.csv")
df_unprocessed = pd.read_csv("/content/e-commerce-marketing-and-sales-revenue-prediction/train.csv")
df.head()

,id,date,region,channel,product_category,customer_segment,ad_spend,price,discount_rate,market_reach,impressions,click_through_rate,competition_index,seasonality_index,campaign_duration_days,customer_lifetime_value,sales_revenue
0,1,2011-12-05 11:31:00,Nort,Search,General,Standard,9.00,0.75,0.2782,32.0,817,0.0010,3.34,1.000000,30.0,816.49,119.767811
1,2,2011-04-27 14:01:00,North,Social Media,General,Premium,3.35,3.35,0.0912,61.0,2289,0.0640,4.44,0.366025,90.0,1723.16,119.404661
2,3,2010-11-09 15:20:00,North,Affiliate,General,Budget,2.55,2.55,0.1997,461.0,14697,0.1508,3.31,0.366025,21.0,1151.74,132.009747
3,4,2010-10-03 15:24:00,North,Affiliate,Storage,Premium,2.95,2.95,0.4767,744.0,17578,0.1965,2.87,-0.366025,90.0,3585.85,154.511756
4,5,2011-10-14 09:28:00,North,Search,Lighting,Premium,15.00,1.25,0.3536,226.0,6280,0.0200,7.40,-0.366025,90.0,502.28,128.059924


## Correcting Date formating

In [6]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [7]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek

## Region Typo Mistake

In [8]:
df['region'] = df['region'].str.lower().str.strip()

df['region'] = df['region'].replace({
    'nort': 'north',
    'norht': 'north'
})

In [9]:
df["channel"] = df["channel"].str.lower().str.strip()

## Fill null Enteries

In [10]:
df['ad_spend'] = df['ad_spend'].fillna(df['ad_spend'].median())
df['discount_rate'] = df['discount_rate'].fillna(df['discount_rate'].median())
df['market_reach'] = df['market_reach'].fillna(df['market_reach'].median())
df['click_through_rate'] = df['click_through_rate'].fillna(df['click_through_rate'].median())
df['competition_index'] = df['competition_index'].fillna(df['competition_index'].median())
df['campaign_duration_days'] = df['campaign_duration_days'].fillna(df['campaign_duration_days'].median())
df['customer_lifetime_value'] = df['customer_lifetime_value'].fillna(df['customer_lifetime_value'].median())

## Detect Outliers

In [12]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1

df = df[(df["price"] >= Q1 - 1.5*IQR) &
        (df["price"] <= Q3 + 1.5*IQR)]

In [13]:
Q1 = df['ad_spend'].quantile(0.25)
Q3 = df['ad_spend'].quantile(0.75)

IQR = Q3 - Q1

df = df[(df['ad_spend'] >= Q1 - 1.5*IQR) &
        (df['ad_spend'] <= Q3 + 1.5*IQR)]

In [14]:
Q1 = df['customer_lifetime_value'].quantile(0.25)
Q3 = df['customer_lifetime_value'].quantile(0.75)

IQR = Q3 - Q1

df = df[(df['customer_lifetime_value'] >= Q1 - 1.5*IQR) &
        (df['customer_lifetime_value'] <= Q3 + 1.5*IQR)]

## Remove Impossible Values

In [15]:
df = df[df['click_through_rate'] <= 1]

In [16]:
df = df[df['impressions'] >= 0]

## Feature Engineering

In [17]:
df['ad_efficiency'] = df['customer_lifetime_value'] / df['ad_spend']

In [18]:
df['cost_per_impression'] = df['ad_spend'] / df['impressions']

In [19]:
df['engagement'] = df['click_through_rate'] * df['impressions']

## Encoding Categorical Values

In [20]:
df = pd.get_dummies(
    df,
    columns=['region','channel','product_category','customer_segment'],
    drop_first=True
)

## Normalization

In [21]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df['ad_spend'] = scaler.fit_transform(df[['ad_spend']])
df['customer_lifetime_value'] = scaler.fit_transform(df[['customer_lifetime_value']])
df['campaign_duration_days'] = scaler.fit_transform(df[['campaign_duration_days']])
df['competition_index'] = scaler.fit_transform(df[['competition_index']])
df['click_through_rate'] = scaler.fit_transform(df[['click_through_rate']])
df['market_reach'] = scaler.fit_transform(df[['market_reach']])

## Drop unnessasery columns

In [29]:
df = df.drop(columns=['id','date'])
df_unprocessed = df_unprocessed.drop(columns=['id','date'])

## Train Test Split

In [23]:
X = df.drop('customer_lifetime_value', axis=1)
y = df['customer_lifetime_value']

In [41]:
df_unprocessed.dropna(subset=['customer_lifetime_value'], inplace=True)
A = df_unprocessed.drop('customer_lifetime_value', axis=1)
B = df_unprocessed['customer_lifetime_value']

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [42]:
from sklearn.model_selection import train_test_split

A_train, A_test, B_train, B_test = train_test_split(
    A, B, test_size=0.2, random_state=42
)

## Evaluation

In [26]:
from sklearn.ensemble import RandomForestRegressor

baseline_model = RandomForestRegressor(random_state=42)

baseline_model.fit(X_train, y_train)

y_pred_base = baseline_model.predict(X_test)

In [43]:
from sklearn.ensemble import RandomForestRegressor

Unprocessed_baseline_model = RandomForestRegressor(random_state=42)

Unprocessed_baseline_model.fit(A_train, B_train)

B_pred_base = Unprocessed_baseline_model.predict(A_test)

In [36]:
df_unprocessed['region'] = df_unprocessed['region'].str.lower().str.strip()
df_unprocessed['region'] = df_unprocessed['region'].replace({
    'nort': 'north',
    'norht': 'north'
})
df_unprocessed["channel"] = df_unprocessed["channel"].str.lower().str.strip()

In [37]:
df_unprocessed = pd.get_dummies(
    df_unprocessed,
    columns=['region','channel','product_category','customer_segment'],
    drop_first=True
)

In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_base = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_base = r2_score(y_test, y_pred_base)

print("Baseline MAE:", mae_base)
print("Baseline RMSE:", rmse_base)
print("Baseline R2:", r2_base)

Baseline MAE: 0.02213832188350277
Baseline RMSE: 0.0728970461363842
Baseline R2: 0.9947180166589844


In [44]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_base = mean_absolute_error(B_test, B_pred_base)
rmse_base = np.sqrt(mean_squared_error(B_test, B_pred_base))
r2_base = r2_score(B_test, B_pred_base)

print("Baseline MAE:", mae_base)
print("Baseline RMSE:", rmse_base)
print("Baseline R2:", r2_base)

Baseline MAE: 13964.99918767671
Baseline RMSE: 34768.05126251062
Baseline R2: 0.003514350908996544
